# Gradient-boosted trees for readmission risk: worked solution

Worked solution for the exercise in the lesson **[Gradient-boosted trees for readmission risk](https://ramkumarmd.com/medical-ai/machine-learning/gradient-boosting-readmission)** (*Machine learning* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](04-gradient-boosting-readmission.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Gradient boosting builds many small decision trees in sequence, each correcting the errors of the ones before, and often performs very well on tabular data. It can find non-linear relationships and interactions without being told about them. For hospital readmission, as for many clinical problems, the gain over a well-specified logistic regression is often small, so compare them fairly, tune with early stopping, and check calibration as well as AUROC.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q shap

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A synthetic readmission dataset

Synthetic discharges with non-linear effects and an interaction built in.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(8)
n = 12000
df = pd.DataFrame({
    "age": rng.integers(20, 95, n),
    "length_of_stay": rng.gamma(2.0, 2.5, n).round(1),
    "prior_admissions": rng.poisson(0.8, n),
    "sodium": rng.normal(138, 4, n).round(),
    "heart_failure": rng.binomial(1, 0.15, n),
    "lives_alone": rng.binomial(1, 0.25, n),
})
logit = (-2.6 + 0.35 * np.minimum(df.prior_admissions, 4)
         + 0.04 * np.maximum(df.age - 65, 0)
         + 0.9 * (np.abs(df.sodium - 139) > 6)        # risk at both low and high sodium
         + 0.6 * df.heart_failure
         + 0.8 * df.heart_failure * df.lives_alone    # interaction
         + 0.05 * df.length_of_stay)
df["readmit_30d"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
print(f"{n} synthetic discharges, 30-day readmission {df.readmit_30d.mean():.1%}")

# 12000 synthetic discharges, 30-day readmission 18.8%

### Boosting versus logistic regression

Continuing from the previous block: both models on the same split; boosting uses early stopping.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split

features = ["age", "length_of_stay", "prior_admissions", "sodium",
            "heart_failure", "lives_alone"]
train, test = train_test_split(df, test_size=0.3, random_state=0, stratify=df.readmit_30d)

models = {
    "Logistic regression": LogisticRegression(max_iter=2000),
    "Gradient boosting": HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=500, max_leaf_nodes=15,
        early_stopping=True, validation_fraction=0.2, random_state=0),
}
for name, model in models.items():
    model.fit(train[features], train.readmit_30d)
    p = model.predict_proba(test[features])[:, 1]
    print(f"{name:<20} AUROC {roc_auc_score(test.readmit_30d, p):.3f}   "
          f"Brier {brier_score_loss(test.readmit_30d, p):.4f}")
print(f"Boosting stopped after {models['Gradient boosting'].n_iter_} trees")

# Logistic regression  AUROC 0.646   Brier 0.1446
# Gradient boosting    AUROC 0.655   Brier 0.1438
# Boosting stopped after 58 trees

### A fair comparison includes a well-specified regression

Continuing from the previous block: logistic regression with clinical terms added.

In [ ]:
def add_terms(d):
    return d.assign(
        sodium_abnormal=(np.abs(d.sodium - 139) > 6).astype(int),
        age_over_65=np.maximum(d.age - 65, 0),
        hf_alone=d.heart_failure * d.lives_alone,
    )
engineered = features + ["sodium_abnormal", "age_over_65", "hf_alone"]
lr2 = LogisticRegression(max_iter=2000).fit(add_terms(train)[engineered], train.readmit_30d)
p = lr2.predict_proba(add_terms(test)[engineered])[:, 1]
print(f"Logistic + clinical terms AUROC {roc_auc_score(test.readmit_30d, p):.3f}   "
      f"Brier {brier_score_loss(test.readmit_30d, p):.4f}")

# Logistic + clinical terms AUROC 0.674   Brier 0.1408

### What the model relies on: permutation importance

Continuing from the 'Boosting versus logistic regression' block.

In [ ]:
from sklearn.inspection import permutation_importance

gb = models["Gradient boosting"]
result = permutation_importance(gb, test[features], test.readmit_30d,
                                scoring="roc_auc", n_repeats=10, random_state=0)
importance = pd.Series(result.importances_mean, index=features)
print(importance.sort_values(ascending=False).round(3))

# age                 0.063
# sodium              0.023
# heart_failure       0.023
# prior_admissions    0.020
# length_of_stay      0.006
# lives_alone         0.002

## Exercise

**Goal:** Compare gradient boosting with logistic regression for 30-day readmission, fairly and on several metrics.

**Data:** Synthetic discharges generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the three models and their AUROC and Brier scores.

In [ ]:
y_test = test.readmit_30d.to_numpy()
fitted = {"Logistic regression": models["Logistic regression"],
          "Gradient boosting": models["Gradient boosting"]}
predictions = {name: m.predict_proba(test[features])[:, 1] for name, m in fitted.items()}
predictions["Logistic + clinical terms"] = lr2.predict_proba(add_terms(test)[engineered])[:, 1]
for name, p in predictions.items():
    print(f"{name:<26} AUROC {roc_auc_score(y_test, p):.3f}   Brier {brier_score_loss(y_test, p):.4f}")

### Step 2

Add natural cubic splines (sklearn SplineTransformer) for age and sodium to the plain logistic regression and compare.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

spline_lr = make_pipeline(
    ColumnTransformer([
        ("splines", SplineTransformer(n_knots=5, degree=3, extrapolation="linear"), ["age", "sodium"]),
        ("rest", StandardScaler(), ["length_of_stay", "prior_admissions", "heart_failure", "lives_alone"]),
    ]),
    LogisticRegression(max_iter=2000),
)
spline_lr.fit(train[features], train.readmit_30d)
predictions["Logistic + splines"] = spline_lr.predict_proba(test[features])[:, 1]
p = predictions["Logistic + splines"]
print(f"Logistic + splines         AUROC {roc_auc_score(y_test, p):.3f}   Brier {brier_score_loss(y_test, p):.4f}")

scikit-learn's SplineTransformer makes B-splines, not natural (restricted) cubic splines; with
extrapolation="linear" they behave similarly at the edges. The splines let the model learn the U-shaped
sodium risk and the rise after 65 without being told, recovering about two-thirds of the gap between the
plain model and the hand-built clinical terms.
What they cannot find is the heart-failure-by-living-alone interaction, which still has to be added.

### Step 3

Tune the boosting model's learning rate and leaf count with cross-validation on the training set only.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

search = GridSearchCV(
    HistGradientBoostingClassifier(max_iter=300, early_stopping=False, random_state=0),
    param_grid={"learning_rate": [0.02, 0.05, 0.1], "max_leaf_nodes": [4, 8, 15, 31]},
    scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=0), n_jobs=-1,
)
search.fit(train[features], train.readmit_30d)       # training data only: the test set stays untouched
cv = pd.DataFrame(search.cv_results_)[["param_learning_rate", "param_max_leaf_nodes", "mean_test_score"]]
print(cv.sort_values("mean_test_score", ascending=False).head(5).round(3).to_string(index=False))
tuned = search.best_estimator_
predictions["Gradient boosting, tuned"] = tuned.predict_proba(test[features])[:, 1]
p = predictions["Gradient boosting, tuned"]
print(f"\nBest {search.best_params_}: test AUROC {roc_auc_score(y_test, p):.3f}, Brier {brier_score_loss(y_test, p):.4f}")

### Step 4

Draw calibration plots for all models; recalibrate the boosted model with isotonic regression if needed.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay

iso = CalibratedClassifierCV(
    HistGradientBoostingClassifier(**search.best_params_, max_iter=300, early_stopping=False, random_state=0),
    method="isotonic", cv=5).fit(train[features], train.readmit_30d)
predictions["Gradient boosting, tuned + isotonic"] = iso.predict_proba(test[features])[:, 1]

fig, ax = plt.subplots(figsize=(6, 5))
for name, p in predictions.items():
    CalibrationDisplay.from_predictions(y_test, p, n_bins=10, strategy="quantile", ax=ax, name=name)
ax.set_title("Calibration on the test set")
plt.show()

for name, p in predictions.items():
    print(f"{name:<36} Brier {brier_score_loss(y_test, p):.4f}   mean predicted {p.mean():.3f} (observed {y_test.mean():.3f})")

All the models are reasonably calibrated here, because training and test data come from the same
population; isotonic recalibration changes the boosted model's Brier score only slightly. Recalibration
matters most when a model is moved to a new setting (lesson 8) or when it overfits.

### Step 5

Compute SHAP values for the boosted model and check whether they reveal the heart-failure-by-living-alone interaction that permutation importance missed.

In [ ]:
import shap

sv = shap.TreeExplainer(tuned)(test[features])
t = test.reset_index(drop=True)
alone_shap = pd.Series(sv[:, "lives_alone"].values)
print("Mean SHAP value for lives_alone = 1, by heart failure:")
print(alone_shap[t.lives_alone == 1].groupby(t.heart_failure[t.lives_alone == 1]).mean().round(2).to_string())

shap.plots.scatter(sv[:, "lives_alone"], color=sv[:, "heart_failure"])

Permutation importance gave lives_alone almost no importance, because on its own it matters only for the
15% of patients with heart failure. SHAP values, which are calculated patient by patient, show the
interaction directly: living alone adds clearly more risk for patients with heart failure than for those
without. The scatter plot colours each point by heart failure, so the two groups separate visibly.

---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/gradient-boosting-readmission](https://ramkumarmd.com/medical-ai/machine-learning/gradient-boosting-readmission). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.